# ♟️ Chess RL Training: AlphaZero / Lc0-Style Pipeline

**Architecture inspired by:** Leela Chess Zero (Lc0) & AlphaZero  
**Reference:** *Methods for analysing Leela Chess Zero* — Panguel & Poupart, 2025

## Pipeline Overview
1. Deep Residual CNN (Lc0-style) with Policy + Value heads
2. PUCT-based Monte Carlo Tree Search (AlphaZero variant)
3. Self-play data generation
4. Iterative training loop with replay buffer
5. Checkpointing, game PNG export, rich logging
6. Evaluation vs random/shallow MCTS agents


## 1. Setup & Imports

In [ ]:
# ─── Core ───────────────────────────────────────────────────────────────────
import os, sys, time, math, json, random, copy, warnings
from pathlib import Path
from collections import deque, defaultdict
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional

import numpy as np
import matplotlib
matplotlib.use('Agg')  # headless backend
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ─── PyTorch ────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# ─── Chess ──────────────────────────────────────────────────────────────────
# Install the python-chess library if not already installed
try:
    import chess
    import chess.svg
except ImportError:
    print("Installing 'python-chess' library...")
    !pip install python-chess
    import chess
    import chess.svg

warnings.filterwarnings('ignore')

# ─── Device ─────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️  Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# ─── Reproducibility ────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE.type == 'cuda':
    torch.cuda.manual_seed_all(SEED)

print('✅ Imports complete')


## 2. Google Drive Mounting (Colab) & Output Paths

In [ ]:
# ─── Detect environment ─────────────────────────────────────────────────────
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/chess_rl')
    print('✅ Google Drive mounted')
else:
    BASE_DIR = Path('./chess_rl_output')
    print('💡 Running locally — outputs saved to ./chess_rl_output')

BASE_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = BASE_DIR / 'checkpoints'
CHECKPOINT_DIR.mkdir(exist_ok=True)
LOG_DIR = BASE_DIR / 'logs'
LOG_DIR.mkdir(exist_ok=True)

print(f'📁 Base directory: {BASE_DIR}')

Mounted at /content/drive
✅ Google Drive mounted
📁 Base directory: /content/drive/MyDrive/chess_rl


## 3. Configuration

In [ ]:
@dataclass
class Config:
    # ── Network ──────────────────────────────────────────────────────────────
    num_res_blocks: int = 10        # residual tower depth (10–20 recommended)
    num_filters:    int = 128       # channels (256 in full Lc0; 128 for speed)
    input_planes:   int = 112       # 112×8×8 input tensor
    policy_size:    int = 4672      # AlphaZero move encoding space

    # ── MCTS ─────────────────────────────────────────────────────────────────
    num_simulations:  int   = 100   # MCTS sims per move
    c_puct:           float = 1.5   # exploration constant
    dirichlet_alpha:  float = 0.3   # noise concentration at root
    dirichlet_eps:    float = 0.25  # fraction of Dirichlet noise mixed in
    temp_threshold:   int   = 15    # moves before temperature drops to ~0
    temp_init:        float = 1.0
    temp_final:       float = 0.1

    # ── Self-play ────────────────────────────────────────────────────────────
    games_per_iter:    int = 4      # self-play games per iteration
    max_game_length:   int = 200    # move cap to avoid infinite games

    # ── Training ─────────────────────────────────────────────────────────────
    num_iterations:    int   = 100
    batch_size:        int   = 256
    epochs_per_iter:   int   = 5
    lr:                float = 1e-3
    weight_decay:      float = 1e-4
    grad_clip_norm:    float = 5.0
    replay_buffer_size:int   = 50_000

    # ── Logging / Checkpointing ───────────────────────────────────────────────
    checkpoint_every:  int = 1
    log_every:         int = 5

    # ── Stability thresholds ─────────────────────────────────────────────────
    entropy_warn_threshold: float = 0.5   # warn if policy entropy < this
    value_collapse_eps:     float = 0.05  # warn if |value| < this always

CFG = Config()
print('⚙️  Configuration loaded:')
for k, v in CFG.__dict__.items():
    print(f'   {k:30s} = {v}')

⚙️  Configuration loaded:
   num_res_blocks                 = 10
   num_filters                    = 128
   input_planes                   = 112
   policy_size                    = 4672
   num_simulations                = 100
   c_puct                         = 1.5
   dirichlet_alpha                = 0.3
   dirichlet_eps                  = 0.25
   temp_threshold                 = 15
   temp_init                      = 1.0
   temp_final                     = 0.1
   games_per_iter                 = 4
   max_game_length                = 200
   num_iterations                 = 100
   batch_size                     = 256
   epochs_per_iter                = 5
   lr                             = 0.001
   weight_decay                   = 0.0001
   grad_clip_norm                 = 5.0
   replay_buffer_size             = 50000
   checkpoint_every               = 20
   log_every                      = 5
   entropy_warn_threshold         = 0.5
   value_collapse_eps             = 0.05


## 4. Chess Board Encoding (112 × 8 × 8 Tensor)

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
#  Board → tensor encoding identical to Lc0 / AlphaZero specification:
#    Planes 0–11:   current position (6 white piece types + 6 black)
#    Planes 12–23:  position T-1  (one move back)
#    ...repeats for 8 history steps  → 8 × 13 = 104 planes
#    Plane 104: white queenside castling
#    Plane 105: white kingside castling
#    Plane 106: black queenside castling
#    Plane 107: black kingside castling
#    Plane 108: side to move (1 = black)
#    Plane 109: halfmove clock (normalised to [0,1])
#    Plane 110: zeros (unused)
#    Plane 111: ones  (constant)
# ──────────────────────────────────────────────────────────────────────────────

PIECE_TO_PLANE = {
    (chess.PAWN,   chess.WHITE): 0,
    (chess.KNIGHT, chess.WHITE): 1,
    (chess.BISHOP, chess.WHITE): 2,
    (chess.ROOK,   chess.WHITE): 3,
    (chess.QUEEN,  chess.WHITE): 4,
    (chess.KING,   chess.WHITE): 5,
    (chess.PAWN,   chess.BLACK): 6,
    (chess.KNIGHT, chess.BLACK): 7,
    (chess.BISHOP, chess.BLACK): 8,
    (chess.ROOK,   chess.BLACK): 9,
    (chess.QUEEN,  chess.BLACK): 10,
    (chess.KING,   chess.BLACK): 11,
}


def board_to_planes(board: chess.Board) -> np.ndarray:
    """Return (12,8,8) float32 array for one board position."""
    planes = np.zeros((12, 8, 8), dtype=np.float32)
    for sq, piece in board.piece_map().items():
        rank, file = divmod(sq, 8)
        plane_idx = PIECE_TO_PLANE[(piece.piece_type, piece.color)]
        planes[plane_idx, rank, file] = 1.0
    return planes


def encode_board(board: chess.Board,
                 history: Optional[List[chess.Board]] = None) -> np.ndarray:
    """
    Encode board + history into a (112, 8, 8) float32 tensor.
    'history' should be ordered oldest→newest (not including current board).
    """
    tensor = np.zeros((112, 8, 8), dtype=np.float32)

    # Build ordered list: current + up to 7 previous
    boards = [board]
    if history:
        boards = list(history[-7:]) + [board]
    boards = boards[-8:]  # cap at 8

    # Fill planes 0–103  (newest first)
    for i, b in enumerate(reversed(boards)):
        offset = i * 13
        tensor[offset:offset+12] = board_to_planes(b)
        # plane 12 of each history slot: repetition count (simplified)
        tensor[offset + 12] = float(b.is_repetition(2))

    # Planes 104–111: game metadata
    tensor[104] = float(board.has_queenside_castling_rights(chess.WHITE))
    tensor[105] = float(board.has_kingside_castling_rights(chess.WHITE))
    tensor[106] = float(board.has_queenside_castling_rights(chess.BLACK))
    tensor[107] = float(board.has_kingside_castling_rights(chess.BLACK))
    tensor[108] = float(board.turn == chess.BLACK)
    tensor[109] = min(board.halfmove_clock / 100.0, 1.0)
    tensor[110] = 0.0
    tensor[111] = 1.0

    return tensor


# ─── Move encoding / decoding ───────────────────────────────────────────────
# AlphaZero uses a 4672-dimensional policy vector (73 move types × 64 squares).
# We use a compact but consistent hash: move UCI → fixed integer index.

_MOVE_INDEX: Dict[str, int] = {}
_INDEX_MOVE: Dict[int, str] = {}

def _build_move_index():
    """Pre-build a full AlphaZero-style move index."""
    idx = 0
    for from_sq in chess.SQUARES:
        for to_sq in chess.SQUARES:
            if from_sq == to_sq:
                continue
            for promo in [None, chess.QUEEN, chess.ROOK,
                          chess.BISHOP, chess.KNIGHT]:
                m = chess.Move(from_sq, to_sq, promotion=promo)
                uci = m.uci()
                if uci not in _MOVE_INDEX:
                    _MOVE_INDEX[uci] = idx
                    _INDEX_MOVE[idx] = uci
                    idx += 1
    # Pad to exactly CFG.policy_size
    # This loop is problematic if CFG.policy_size is smaller than idx
    # It should be dynamically adjusted after determining actual moves.
    while idx < CFG.policy_size:
        _INDEX_MOVE[idx] = None
        idx += 1

_build_move_index()
ACTUAL_POLICY_SIZE = len(_MOVE_INDEX)
# Dynamically update CFG.policy_size to match the actual number of encoded moves
CFG.policy_size = ACTUAL_POLICY_SIZE
print(f'📐 Move index built: {ACTUAL_POLICY_SIZE} unique moves encoded')


def move_to_index(move: chess.Move) -> int:
    return _MOVE_INDEX.get(move.uci(), 0)


def legal_move_mask(board: chess.Board) -> np.ndarray:
    """Return a boolean mask over the policy vector for legal moves."""
    mask = np.zeros(CFG.policy_size, dtype=np.float32)
    for move in board.legal_moves:
        idx = move_to_index(move)
        if idx < CFG.policy_size: # This check caused the issue
            mask[idx] = 1.0
    return mask


# ─── Quick sanity check ─────────────────────────────────────────────────────
board = chess.Board()
t = encode_board(board)
assert t.shape == (112, 8, 8), f'Tensor shape mismatch: {t.shape}'
mask = legal_move_mask(board)
assert mask.sum() == 20, f'Expected 20 legal opening moves, got {mask.sum()}'
print(f'✅ Board encoding: shape={t.shape}, opening legal moves={int(mask.sum())}')

📐 Move index built: 20160 unique moves encoded
✅ Board encoding: shape=(112, 8, 8), opening legal moves=20


## 5. Neural Network (Lc0-Style Residual CNN)

In [ ]:
class ResidualBlock(nn.Module):
    """Standard AlphaZero / Lc0 residual block."""
    def __init__(self, num_filters: int):
        super().__init__()
        self.conv1 = nn.Conv2d(num_filters, num_filters, 3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(num_filters)
        self.conv2 = nn.Conv2d(num_filters, num_filters, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(num_filters)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        return F.relu(x + residual)  # skip connection


class ChessNet(nn.Module):
    """
    Lc0-inspired deep residual CNN.

    Architecture:
      Input  : (batch, 112, 8, 8)
      Trunk  : Conv(3×3, F) → BN → ReLU → N × ResBlock
      Policy : Conv(1×1,2) → BN → ReLU → FC → softmax(4672)
      Value  : Conv(1×1,1) → BN → ReLU → FC(256) → ReLU → FC(1) → tanh
    """
    def __init__(self, cfg: Config):
        super().__init__()
        F_ = cfg.num_filters

        # ── Input stem ──────────────────────────────────────────────────────
        self.input_conv = nn.Conv2d(cfg.input_planes, F_, 3, padding=1, bias=False)
        self.input_bn   = nn.BatchNorm2d(F_)

        # ── Residual tower ───────────────────────────────────────────────────
        self.res_tower = nn.ModuleList(
            [ResidualBlock(F_) for _ in range(cfg.num_res_blocks)]
        )

        # ── Policy head ──────────────────────────────────────────────────────
        self.policy_conv = nn.Conv2d(F_, 2, 1, bias=False)
        self.policy_bn   = nn.BatchNorm2d(2)
        self.policy_fc   = nn.Linear(2 * 8 * 8, cfg.policy_size)

        # ── Value head ───────────────────────────────────────────────────────
        self.value_conv  = nn.Conv2d(F_, 1, 1, bias=False)
        self.value_bn    = nn.BatchNorm2d(1)
        self.value_fc1   = nn.Linear(8 * 8, 256)
        self.value_fc2   = nn.Linear(256, 1)

        # Weight initialisation
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias,   0)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.constant_(m.bias, 0)

    def forward(self, x: torch.Tensor):
        # Trunk
        x = F.relu(self.input_bn(self.input_conv(x)))
        for block in self.res_tower:
            x = block(x)

        # Policy head
        p = F.relu(self.policy_bn(self.policy_conv(x)))
        p = p.view(p.size(0), -1)
        p = self.policy_fc(p)          # logits, shape (B, 4672)

        # Value head
        v = F.relu(self.value_bn(self.value_conv(x)))
        v = v.view(v.size(0), -1)
        v = F.relu(self.value_fc1(v))
        v = torch.tanh(self.value_fc2(v))  # scalar in (-1, 1)

        return p, v

    @torch.no_grad()
    def predict(self, board: chess.Board,
                history: Optional[List[chess.Board]] = None
                ) -> Tuple[np.ndarray, float]:
        """Inference for a single board — returns (policy_probs, value)."""
        self.eval()
        t = encode_board(board, history)
        x = torch.tensor(t, dtype=torch.float32).unsqueeze(0).to(DEVICE)
        logits, v = self(x)

        # Mask illegal moves then softmax
        mask = legal_move_mask(board)
        mask_t = torch.tensor(mask, dtype=torch.float32).to(DEVICE)
        logits = logits.squeeze(0)
        logits = logits - (1 - mask_t) * 1e9   # large negative for illegal
        probs = torch.softmax(logits, dim=0).cpu().numpy()
        value = float(v.item())
        return probs, value


# ─── Instantiate ─────────────────────────────────────────────────────────────
net = ChessNet(CFG).to(DEVICE)
total_params = sum(p.numel() for p in net.parameters())
trainable_params = sum(p.numel() for p in net.parameters() if p.requires_grad)
print(f'🧠 ChessNet instantiated on {DEVICE}')
print(f'   Total params:     {total_params:,}')
print(f'   Trainable params: {trainable_params:,}')

# Quick forward pass sanity check
_dummy = torch.zeros(2, 112, 8, 8).to(DEVICE)
_p, _v = net(_dummy)
assert _p.shape == (2, CFG.policy_size)
assert _v.shape == (2, 1)
print(f'✅ Forward pass OK — policy shape={tuple(_p.shape)}, value shape={tuple(_v.shape)}')

🧠 ChessNet instantiated on cuda
   Total params:     5,701,447
   Trainable params: 5,701,447
✅ Forward pass OK — policy shape=(2, 20160), value shape=(2, 1)


## 6. Monte Carlo Tree Search (PUCT / AlphaZero)

In [ ]:
class MCTSNode:
    """A single node in the MCTS tree."""
    __slots__ = ('board', 'parent', 'move', 'children',
                 'N', 'W', 'Q', 'P', 'is_expanded', 'history')

    def __init__(self, board: chess.Board,
                 parent: Optional['MCTSNode'] = None,
                 move: Optional[chess.Move] = None,
                 prior: float = 0.0,
                 history: Optional[List[chess.Board]] = None):
        self.board       = board.copy()
        self.parent      = parent
        self.move        = move
        self.children: Dict[chess.Move, 'MCTSNode'] = {}
        self.N = 0          # visit count
        self.W = 0.0        # total action value
        self.Q = 0.0        # mean action value
        self.P = prior      # prior probability from policy head
        self.is_expanded = False
        self.history = history or []


class MCTS:
    """AlphaZero-style PUCT Monte Carlo Tree Search."""

    def __init__(self, net: ChessNet, cfg: Config):
        self.net = net
        self.cfg = cfg

    # ── UCB formula (PUCT) ───────────────────────────────────────────────────
    def _ucb(self, node: MCTSNode, child: MCTSNode) -> float:
        """
        UCB(s,a) = Q(s,a) + c_puct * P(s,a) * sqrt(N(s)) / (1 + N(s,a))
        """
        u = (self.cfg.c_puct * child.P *
             math.sqrt(node.N + 1) / (1 + child.N))
        return child.Q + u

    # ── Select ───────────────────────────────────────────────────────────────
    def _select(self, node: MCTSNode) -> MCTSNode:
        """Descend tree following PUCT until a leaf."""
        while node.is_expanded and node.children:
            node = max(node.children.values(),
                       key=lambda c: self._ucb(node, c))
        return node

    # ── Expand ───────────────────────────────────────────────────────────────
    def _expand(self, node: MCTSNode) -> float:
        """
        Query neural network at this node, create child nodes with priors.
        Returns the value estimate from the value head.
        """
        board = node.board

        if board.is_game_over():
            return self._terminal_value(board)

        probs, value = self.net.predict(board, node.history)
        node.is_expanded = True

        for move in board.legal_moves:
            idx = move_to_index(move)
            prior = float(probs[idx]) if idx < len(probs) else 1e-8
            child_board = board.copy()
            child_board.push(move)
            child_history = node.history[-7:] + [board]
            node.children[move] = MCTSNode(
                board=child_board,
                parent=node,
                move=move,
                prior=prior,
                history=child_history
            )
        return value

    # ── Backpropagate ────────────────────────────────────────────────────────
    def _backup(self, node: MCTSNode, value: float):
        """Propagate value up to root, flipping sign at each level."""
        while node is not None:
            node.N += 1
            node.W += value
            node.Q  = node.W / node.N
            value  = -value   # opponent's perspective
            node   = node.parent

    # ── Terminal value ───────────────────────────────────────────────────────
    @staticmethod
    def _terminal_value(board: chess.Board) -> float:
        result = board.result()
        if result == '1-0':
            return 1.0 if board.turn == chess.BLACK else -1.0
        if result == '0-1':
            return 1.0 if board.turn == chess.WHITE else -1.0
        return 0.0  # draw

    # ── Dirichlet noise at root ───────────────────────────────────────────────
    def _add_dirichlet_noise(self, root: MCTSNode):
        if not root.children:
            return
        n = len(root.children)
        noise = np.random.dirichlet([self.cfg.dirichlet_alpha] * n)
        eps = self.cfg.dirichlet_eps
        for child, eta in zip(root.children.values(), noise):
            child.P = (1 - eps) * child.P + eps * eta

    # ── Run full MCTS search ─────────────────────────────────────────────────
    def search(self, board: chess.Board,
               history: Optional[List[chess.Board]] = None,
               add_noise: bool = True) -> MCTSNode:
        """Run N simulations and return the root node."""
        root = MCTSNode(board, history=history or [])
        # Expand root first
        self._backup(root, self._expand(root))
        if add_noise:
            self._add_dirichlet_noise(root)

        for _ in range(self.cfg.num_simulations - 1):
            leaf = self._select(root)
            value = self._expand(leaf)
            self._backup(leaf, value)

        return root

    # ── Extract policy target ────────────────────────────────────────────────
    def get_policy_target(self, root: MCTSNode,
                          temperature: float = 1.0) -> np.ndarray:
        """
        Convert visit counts to a policy target.
        policy_target[i] = N(s, a_i)^(1/T) / sum_j N(s, a_j)^(1/T)
        """
        policy = np.zeros(self.cfg.policy_size, dtype=np.float32)
        if not root.children:
            return policy

        visits = {move: child.N for move, child in root.children.items()}
        total = sum(visits.values()) + 1e-10

        if temperature < 0.01:  # greedy
            best_move = max(visits, key=visits.get)
            policy[move_to_index(best_move)] = 1.0
        else:
            temp_visits = {m: v**(1.0/temperature) for m, v in visits.items()}
            total_t = sum(temp_visits.values()) + 1e-10
            for move, v in temp_visits.items():
                idx = move_to_index(move)
                if idx < self.cfg.policy_size:
                    policy[idx] = v / total_t
        return policy

    # ── Select move ──────────────────────────────────────────────────────────
    def select_move(self, root: MCTSNode,
                    temperature: float = 1.0) -> chess.Move:
        if not root.children:
            raise ValueError('Root has no children — board is terminal?')
        policy = self.get_policy_target(root, temperature)
        legal_indices = [move_to_index(m) for m in root.children]
        legal_probs = np.array([policy[i] for i in legal_indices])
        legal_probs = legal_probs / (legal_probs.sum() + 1e-10)
        chosen_idx = np.random.choice(len(legal_indices), p=legal_probs)
        return list(root.children.keys())[chosen_idx]


print('✅ MCTS defined')

✅ MCTS defined


## 7. Self-Play Game Generation

In [ ]:
@dataclass
class GameRecord:
    """One completed self-play game."""
    states:         List[np.ndarray]  # (112,8,8) tensors
    policy_targets: List[np.ndarray]  # (4672,) MCTS visit distributions
    value_targets:  List[float]       # game outcome from each player's perspective
    moves:          List[str]         # UCI strings
    result:         str               # '1-0', '0-1', '1/2-1/2'
    game_length:    int


class SelfPlayWorker:
    """Runs self-play games and collects training tuples."""

    def __init__(self, net: ChessNet, cfg: Config):
        self.net  = net
        self.cfg  = cfg
        self.mcts = MCTS(net, cfg)

    def play_game(self, verbose: bool = False) -> GameRecord:
        board   = chess.Board()
        history: List[chess.Board] = []
        states, policies, moves = [], [], []
        move_count = 0

        while not board.is_game_over() and move_count < self.cfg.max_game_length:
            temperature = (
                self.cfg.temp_init
                if move_count < self.cfg.temp_threshold
                else self.cfg.temp_final
            )

            # Store current state
            state_tensor = encode_board(board, history)
            states.append(state_tensor)

            # Run MCTS
            root = self.mcts.search(board, history, add_noise=True)
            policy_target = self.mcts.get_policy_target(root, temperature)
            policies.append(policy_target)

            # Select & play move
            move = self.mcts.select_move(root, temperature)
            moves.append(move.uci())
            history.append(board.copy())
            board.push(move)
            move_count += 1

            if verbose and move_count % 10 == 0:
                print(f'  Move {move_count}: {move.uci()} | temp={temperature:.2f}')

        # Determine outcome
        result = board.result()
        if   result == '1-0': z = 1.0
        elif result == '0-1': z = -1.0
        else:                 z = 0.0

        # Assign value targets from each position's perspective
        value_targets = []
        current_z = z
        for i in range(len(states) - 1, -1, -1):
            value_targets.insert(0, current_z)
            current_z = -current_z

        return GameRecord(
            states=states,
            policy_targets=policies,
            value_targets=value_targets,
            moves=moves,
            result=result,
            game_length=move_count,
        )

    def generate_games(self, num_games: int, verbose: bool = False) -> List[GameRecord]:
        games = []
        for i in range(num_games):
            t0 = time.time()
            game = self.play_game(verbose=verbose)
            elapsed = time.time() - t0
            games.append(game)
            print(f'  Game {i+1}/{num_games}: result={game.result}, '
                  f'moves={game.game_length}, time={elapsed:.1f}s')
        return games


print('✅ SelfPlayWorker defined')

✅ SelfPlayWorker defined


## 8. Replay Buffer & Dataset

In [ ]:
class ReplayBuffer:
    """Fixed-size circular replay buffer with deque."""

    def __init__(self, max_size: int):
        self.buffer: deque = deque(maxlen=max_size)

    def add_game(self, game: GameRecord):
        for s, p, v in zip(game.states, game.policy_targets, game.value_targets):
            self.buffer.append((s, p, float(v)))

    def __len__(self):
        return len(self.buffer)

    def sample(self, batch_size: int):
        batch = random.sample(self.buffer, min(batch_size, len(self.buffer)))
        states, policies, values = zip(*batch)
        return (
            np.stack(states),
            np.stack(policies),
            np.array(values, dtype=np.float32)
        )


class ChessDataset(Dataset):
    def __init__(self, states, policies, values):
        self.states   = torch.tensor(states,   dtype=torch.float32)
        self.policies = torch.tensor(policies, dtype=torch.float32)
        self.values   = torch.tensor(values,   dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.states)

    def __getitem__(self, idx):
        return self.states[idx], self.policies[idx], self.values[idx]


print('✅ ReplayBuffer & ChessDataset defined')

✅ ReplayBuffer & ChessDataset defined


## 9. Training Loop with Stability Safeguards

In [ ]:
class Trainer:
    """Manages the network training with loss computation, grad clipping, and LR scheduling."""

    def __init__(self, net: ChessNet, cfg: Config):
        self.net = net
        self.cfg = cfg
        self.optimizer = optim.Adam(
            net.parameters(),
            lr=cfg.lr,
            weight_decay=cfg.weight_decay
        )
        self.scheduler = optim.lr_scheduler.CosineAnnealingLR(
            self.optimizer,
            T_max=cfg.num_iterations,
            eta_min=1e-5
        )
        self.global_step = 0

    def compute_loss(self, states, policy_targets, value_targets):
        """
        total_loss = policy_loss (cross-entropy) + value_loss (MSE)
        """
        policy_logits, value_preds = self.net(states)

        # Policy loss: cross-entropy between MCTS visits and network logits
        # policy_targets are already normalised probability distributions
        log_softmax = F.log_softmax(policy_logits, dim=1)
        policy_loss = -(policy_targets * log_softmax).sum(dim=1).mean()

        # Value loss: MSE
        value_loss = F.mse_loss(value_preds, value_targets)

        total_loss = policy_loss + value_loss
        return total_loss, policy_loss.item(), value_loss.item()

    def train_epoch(self, dataloader) -> Dict:
        self.net.train()
        epoch_stats = defaultdict(float)
        n_batches = 0
        all_grad_norms = []

        for states, policy_targets, value_targets in dataloader:
            states         = states.to(DEVICE)
            policy_targets = policy_targets.to(DEVICE)
            value_targets  = value_targets.to(DEVICE)

            self.optimizer.zero_grad()
            total_loss, pol_loss, val_loss = self.compute_loss(
                states, policy_targets, value_targets
            )
            total_loss.backward()

            # ── Gradient clipping (stability safeguard) ──────────────────────
            grad_norm = torch.nn.utils.clip_grad_norm_(
                self.net.parameters(), self.cfg.grad_clip_norm
            )
            all_grad_norms.append(float(grad_norm))

            self.optimizer.step()
            self.global_step += 1

            epoch_stats['total_loss'] += total_loss.item()
            epoch_stats['policy_loss'] += pol_loss
            epoch_stats['value_loss']  += val_loss
            n_batches += 1

        epoch_stats = {k: v / max(n_batches, 1) for k, v in epoch_stats.items()}
        epoch_stats['grad_norm_mean'] = np.mean(all_grad_norms)
        epoch_stats['grad_norm_max']  = np.max(all_grad_norms)
        return epoch_stats

    def train_on_buffer(self, replay: ReplayBuffer) -> Dict:
        if len(replay) < self.cfg.batch_size:
            print(f'⚠️  Buffer too small ({len(replay)} < {self.cfg.batch_size}), skipping training')
            return {}

        states, policies, values = replay.sample(min(len(replay), 10_000))
        dataset    = ChessDataset(states, policies, values)
        dataloader = DataLoader(dataset, batch_size=self.cfg.batch_size,
                                shuffle=True, drop_last=True)
        all_stats = defaultdict(list)
        for epoch in range(self.cfg.epochs_per_iter):
            stats = self.train_epoch(dataloader)
            for k, v in stats.items():
                all_stats[k].append(v)

        self.scheduler.step()
        return {k: np.mean(v) for k, v in all_stats.items()}


print('✅ Trainer defined')

✅ Trainer defined


## 10. Logging, Statistics & Stability Checks

In [ ]:
class Logger:
    """Tracks metrics and detects instability / collapse conditions."""

    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.history = defaultdict(list)
        self._iter = 0

    def log_iteration(self, iteration: int, games: List[GameRecord],
                      train_stats: Dict, net: ChessNet):
        self._iter = iteration
        print(f'\n{"═"*65}')
        print(f'  Iteration {iteration}')
        print(f'{"═"*65}')

        # ── Game statistics ─────────────────────────────────────────────────
        results = {'1-0': 0, '0-1': 0, '1/2-1/2': 0, '*': 0}
        lengths = []
        all_moves = []
        for g in games:
            results[g.result] = results.get(g.result, 0) + 1
            lengths.append(g.game_length)
            all_moves.extend(g.moves)

        unique_moves = len(set(all_moves))
        print(f'  📊 Game stats:')
        print(f'     Results  : {results}')
        print(f'     Length   : min={min(lengths)}, max={max(lengths)}, '
              f'mean={np.mean(lengths):.1f}')
        print(f'     Unique moves played: {unique_moves} / {len(all_moves)}')

        self.history['game_length_mean'].append(np.mean(lengths))
        self.history['unique_moves'].append(unique_moves)
        self.history['white_wins'].append(results.get('1-0',   0))
        self.history['black_wins'].append(results.get('0-1',   0))
        self.history['draws'].append(results.get('1/2-1/2',   0))

        # ── Training statistics ──────────────────────────────────────────────
        if train_stats:
            print(f'  📉 Training losses:')
            for k, v in train_stats.items():
                print(f'     {k:20s}: {v:.5f}')
            for k, v in train_stats.items():
                self.history[k].append(v)

        # ── Policy entropy (collapse detector) ───────────────────────────────
        entropies = self._measure_policy_entropy(net, games)
        mean_entropy = np.mean(entropies) if entropies else 0.0
        print(f'  🔀 Policy entropy (mean): {mean_entropy:.4f}')
        self.history['policy_entropy'].append(mean_entropy)

        if mean_entropy < self.cfg.entropy_warn_threshold:
            print(f'  ⚠️  WARNING: Policy entropy {mean_entropy:.4f} < '
                  f'threshold {self.cfg.entropy_warn_threshold:.4f}!')
            print(f'      Possible policy collapse — consider increasing Dirichlet noise.')

        # ── Value head collapse detector ─────────────────────────────────────
        value_samples = self._sample_values(net, games[:1])
        if value_samples:
            mean_abs_val = np.mean(np.abs(value_samples))
            print(f'  📏 Mean |value prediction|: {mean_abs_val:.4f}')
            if mean_abs_val < self.cfg.value_collapse_eps:
                print(f'  ⚠️  WARNING: Value head collapsed near zero!')

        # ── Move diversity check ─────────────────────────────────────────────
        if unique_moves < 5:
            print(f'  ⚠️  WARNING: Very low move diversity ({unique_moves} unique moves)!')
            print(f'      Possible repetitive / deterministic play.')

        # ── Gradient norm check ──────────────────────────────────────────────
        if train_stats and train_stats.get('grad_norm_max', 0) > self.cfg.grad_clip_norm * 0.95:
            print(f'  ⚠️  WARNING: Gradient norm near clipping limit — '
                  f'may indicate instability.')

        # ── Repetition detection ──────────────────────────────────────────────
        self._check_repetitions(games)

    def _measure_policy_entropy(self, net: ChessNet,
                                games: List[GameRecord]) -> List[float]:
        """Compute Shannon entropy of policy output on a few sampled positions."""
        net.eval()
        entropies = []
        for game in games[:2]:  # sample from first 2 games
            for state in game.states[:5]:  # first 5 positions
                board = chess.Board()
                probs, _ = net.predict(board)
                # Only consider legal moves
                mask = legal_move_mask(board).astype(bool)
                legal_probs = probs[mask]
                legal_probs = legal_probs / (legal_probs.sum() + 1e-10)
                entropy = -np.sum(legal_probs * np.log(legal_probs + 1e-10))
                entropies.append(entropy)
        return entropies

    def _sample_values(self, net: ChessNet,
                       games: List[GameRecord]) -> List[float]:
        net.eval()
        values = []
        board = chess.Board()
        with torch.no_grad():
            _, v = net.predict(board)
            values.append(v)
        return values

    def _check_repetitions(self, games: List[GameRecord]):
        for i, game in enumerate(games):
            if len(game.moves) > 8:
                recent = game.moves[-8:]
                # detect 2-move repetition (e.g. rook shuffling)
                pairs = [recent[j]+recent[j+2] for j in range(0, 4)]
                if len(set(pairs)) == 1:
                    print(f'  ⚠️  WARNING: Game {i+1} shows possible '
                          f'repetitive shuffling pattern in last 8 moves!')

    def save_history(self, path: Path):
        with open(path / 'training_history.json', 'w') as f:
            json.dump({k: [float(x) for x in v]
                       for k, v in self.history.items()}, f, indent=2)


print('✅ Logger defined')

✅ Logger defined


## 11. Checkpointing & Game PNG Export

In [ ]:
def save_checkpoint(net: ChessNet, optimizer: optim.Optimizer,
                    iteration: int, base_dir: Path):
    path = base_dir / 'checkpoints' / f'model_iter_{iteration}.pt'
    torch.save({
        'iteration':     iteration,
        'model_state':   net.state_dict(),
        'optimizer_state': optimizer.state_dict(),
    }, path)
    print(f'  💾 Checkpoint saved: {path}')


def load_checkpoint(net: ChessNet, optimizer: optim.Optimizer,
                    path: Path) -> int:
    ckpt = torch.load(path, map_location=DEVICE)
    net.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    print(f'  ✅ Checkpoint loaded from {path} (iteration {ckpt["iteration"]})')
    return ckpt['iteration']


def export_game_png(game: GameRecord, iteration: int, base_dir: Path):
    """
    Replay the game and save a matplotlib figure showing the final board
    position as a PNG. Optionally shows a mini-grid of milestone positions.
    """
    try:
        board = chess.Board()
        for uci in game.moves:
            board.push(chess.Move.from_uci(uci))

        # Build board grid image
        n_milestones = min(6, game.game_length)
        milestone_idx = np.linspace(0, game.game_length - 1,
                                    n_milestones, dtype=int)

        fig, axes = plt.subplots(1, n_milestones + 1,
                                 figsize=(3 * (n_milestones + 1), 3.5))
        if n_milestones == 0:
            axes = [axes]

        # Replay to get milestone boards
        replay_board = chess.Board()
        milestone_boards = []
        for idx, uci in enumerate(game.moves):
            if idx in milestone_idx:
                milestone_boards.append((idx, replay_board.copy()))
            replay_board.push(chess.Move.from_uci(uci))

        for ax, (move_num, mb) in zip(axes[:-1], milestone_boards):
            _draw_board_matplotlib(ax, mb)
            ax.set_title(f'Move {move_num}', fontsize=8)

        # Final position
        _draw_board_matplotlib(axes[-1], board)
        axes[-1].set_title(f'Final ({game.result})', fontsize=8)

        plt.suptitle(f'Self-play game — Iteration {iteration} | '
                     f'{game.game_length} moves | Result: {game.result}',
                     fontsize=10, fontweight='bold')
        plt.tight_layout()

        save_path = base_dir / f'game_iter_{iteration}.png'
        plt.savefig(save_path, dpi=100, bbox_inches='tight')
        plt.close(fig)
        print(f'  🖼️  Game PNG saved: {save_path}')

    except Exception as e:
        print(f'  ⚠️  PNG export failed: {e}')


# ─── Simple matplotlib chessboard renderer ───────────────────────────────────
PIECE_UNICODE = {
    chess.PAWN:   ('♙', '♟'), chess.KNIGHT: ('♘', '♞'),
    chess.BISHOP: ('♗', '♝'), chess.ROOK:   ('♖', '♜'),
    chess.QUEEN:  ('♕', '♛'), chess.KING:   ('♔', '♚'),
}

def _draw_board_matplotlib(ax, board: chess.Board):
    colors = ['#F0D9B5', '#B58863']
    for rank in range(8):
        for file in range(8):
            color = colors[(rank + file) % 2]
            ax.add_patch(plt.Rectangle([file, rank], 1, 1, color=color))
            sq = chess.square(file, rank)
            piece = board.piece_at(sq)
            if piece:
                sym = PIECE_UNICODE[piece.piece_type][0 if piece.color else 1]
                ax.text(file + 0.5, rank + 0.5, sym,
                        ha='center', va='center', fontsize=14)
    ax.set_xlim(0, 8); ax.set_ylim(0, 8)
    ax.set_aspect('equal'); ax.axis('off')


print('✅ Checkpointing & PNG export defined')

✅ Checkpointing & PNG export defined


## 12. Visualisation Utilities

In [ ]:
def plot_training_curves(logger: Logger, save_path: Optional[Path] = None):
    h = logger.history
    keys = ['total_loss', 'policy_loss', 'value_loss', 'policy_entropy',
            'game_length_mean', 'unique_moves']
    present = [k for k in keys if h.get(k)]
    if not present:
        print('Not enough data to plot yet.')
        return

    ncols = 3
    nrows = math.ceil(len(present) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
    axes = axes.flatten() if nrows > 1 else [axes] if ncols == 1 else axes.flatten()

    for ax, key in zip(axes, present):
        vals = h[key]
        ax.plot(vals, marker='o', markersize=3, linewidth=1.5)
        ax.set_title(key.replace('_', ' ').title(), fontweight='bold')
        ax.set_xlabel('Iteration')
        ax.grid(True, alpha=0.3)

    for ax in axes[len(present):]:
        ax.axis('off')

    plt.suptitle('Training Curves', fontsize=14, fontweight='bold')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=100, bbox_inches='tight')
        print(f'  📈 Training curves saved: {save_path}')
    plt.show()
    plt.close()


def plot_policy_heatmap(net: ChessNet, board: chess.Board,
                        save_path: Optional[Path] = None):
    """Show a board-level policy heatmap (destination square probability)."""
    probs, value = net.predict(board)
    heatmap = np.zeros((8, 8), dtype=np.float32)
    for move in board.legal_moves:
        idx = move_to_index(move)
        to_sq = move.to_square
        rank, file = divmod(to_sq, 8)
        heatmap[rank, file] += float(probs[idx])

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    # Board
    _draw_board_matplotlib(axes[0], board)
    axes[0].set_title(f'Board Position (value={value:.3f})', fontweight='bold')

    # Heatmap
    im = axes[1].imshow(heatmap, cmap='YlOrRd', origin='lower', vmin=0)
    axes[1].set_xticks(range(8))
    axes[1].set_xticklabels(list('abcdefgh'))
    axes[1].set_yticks(range(8))
    axes[1].set_yticklabels(range(1, 9))
    axes[1].set_title('Policy Heatmap (destination probs)', fontweight='bold')
    plt.colorbar(im, ax=axes[1])

    if save_path:
        plt.savefig(save_path, dpi=100, bbox_inches='tight')
    plt.show()
    plt.close()


def print_top_moves(net: ChessNet, board: chess.Board, top_k: int = 5):
    """Print the top-K moves predicted by the policy head."""
    probs, value = net.predict(board)
    legal_moves = list(board.legal_moves)
    move_probs = [(m, probs[move_to_index(m)]) for m in legal_moves]
    move_probs.sort(key=lambda x: x[1], reverse=True)

    print(f'\n  🎯 Top-{top_k} predicted moves (value={value:+.3f}):')
    for i, (move, prob) in enumerate(move_probs[:top_k], 1):
        bar = '█' * int(prob * 50)
        print(f'    {i}. {move.uci():6s}  {prob:.4f}  {bar}')


print('✅ Visualisation utilities defined')

✅ Visualisation utilities defined


In [ ]:
start_iteration = 1
latest_checkpoint = None

# Find the latest checkpoint
checkpoint_files = sorted(CHECKPOINT_DIR.glob('model_iter_*.pt'), reverse=True)
if checkpoint_files:
    latest_checkpoint = checkpoint_files[0]
    print(f'✨ Found latest checkpoint: {latest_checkpoint}')
    # Instantiate network and optimizer first to load state dict
    net = ChessNet(CFG).to(DEVICE)
    trainer = Trainer(net, CFG)
    start_iteration = load_checkpoint(net, trainer.optimizer, latest_checkpoint) + 1
    print(f'🚀 Resuming training from iteration {start_iteration}')
else:
    print('✨ No existing checkpoint found, starting training from scratch.')
    net = ChessNet(CFG).to(DEVICE)
    trainer = Trainer(net, CFG)

worker  = SelfPlayWorker(net, CFG)
replay  = ReplayBuffer(CFG.replay_buffer_size)
logger  = Logger(CFG)

## 13. Main Training Loop

In [ ]:
# ─── Initialise all components ───────────────────────────────────────────────
net     = ChessNet(CFG).to(DEVICE)
trainer = Trainer(net, CFG)
worker  = SelfPlayWorker(net, CFG)
replay  = ReplayBuffer(CFG.replay_buffer_size)
logger  = Logger(CFG)

print('🚀 Starting AlphaZero / Lc0-style training loop')
print(f'   Iterations: {CFG.num_iterations}')
print(f'   Games/iter: {CFG.games_per_iter}')
print(f'   MCTS sims:  {CFG.num_simulations}')

all_games: List[GameRecord] = []

for iteration in range(1, CFG.num_iterations + 1):
    iter_start = time.time()
    print(f'\n── Iteration {iteration}/{CFG.num_iterations} ──')

    # ─── Step 1: Self-play ───────────────────────────────────────────────────
    print(f'  ♟️  Self-play ({CFG.games_per_iter} games)...')
    games = worker.generate_games(CFG.games_per_iter)
    all_games.extend(games)

    # ─── Step 2: Fill replay buffer ──────────────────────────────────────────
    for game in games:
        replay.add_game(game)
    print(f'  📦 Replay buffer size: {len(replay)}')

    # ─── Step 3: Train ───────────────────────────────────────────────────────
    print(f'  🏋️  Training ({CFG.epochs_per_iter} epochs)...')
    train_stats = trainer.train_on_buffer(replay)

    # ─── Step 4: Logging ─────────────────────────────────────────────────────
    if iteration % CFG.log_every == 0 or iteration == 1:
        logger.log_iteration(iteration, games, train_stats, net)
        print_top_moves(net, chess.Board(), top_k=5)

    # ─── Step 5: Checkpoint + PNG every 20 iters ─────────────────────────────
    if iteration % CFG.checkpoint_every == 0:
        save_checkpoint(net, trainer.optimizer, iteration, BASE_DIR)
        export_game_png(games[-1], iteration, BASE_DIR)
        logger.save_history(LOG_DIR)

    elapsed = time.time() - iter_start
    print(f'  ⏱️  Iteration time: {elapsed:.1f}s')

print('\n🎉 Training complete!')

🚀 Starting AlphaZero / Lc0-style training loop
   Iterations: 100
   Games/iter: 4
   MCTS sims:  100

── Iteration 1/100 ──
  ♟️  Self-play (4 games)...
  Game 1/4: result=0-1, moves=186, time=330.6s
  Game 2/4: result=1-0, moves=39, time=34.3s
  Game 3/4: result=1-0, moves=69, time=79.0s
  Game 4/4: result=1-0, moves=111, time=172.5s
  📦 Replay buffer size: 405
  🏋️  Training (5 epochs)...

═════════════════════════════════════════════════════════════════
  Iteration 1
═════════════════════════════════════════════════════════════════
  📊 Game stats:
     Results  : {'1-0': 3, '0-1': 1, '1/2-1/2': 0, '*': 0}
     Length   : min=39, max=186, mean=101.2
     Unique moves played: 313 / 405
  📉 Training losses:
     total_loss          : 10.25960
     policy_loss         : 9.73528
     value_loss          : 0.52432
     grad_norm_mean      : 5.74618
     grad_norm_max       : 5.74618
  🔀 Policy entropy (mean): 2.9950
  📏 Mean |value prediction|: 0.9292
  ⚠️  WARNING: Gradient norm near cl

In [ ]:
for iteration in range(start_iteration, CFG.num_iterations + 1):

## 14. Post-Training: Curves & Analysis

In [ ]:
# Plot training curves
plot_training_curves(logger, save_path=BASE_DIR / 'training_curves.png')

# Policy heatmap on the starting position
opening_board = chess.Board()
plot_policy_heatmap(net, opening_board,
                    save_path=BASE_DIR / 'policy_heatmap_opening.png')

# Show top moves on opening
print_top_moves(net, chess.Board(), top_k=10)

## 15. Evaluation: vs Random Agent & Shallow MCTS

In [ ]:
class RandomAgent:
    """Plays a uniformly random legal move."""
    def select_move(self, board: chess.Board) -> chess.Move:
        return random.choice(list(board.legal_moves))


class MCTSAgent:
    """Shallow MCTS agent with configurable simulation budget."""
    def __init__(self, net: ChessNet, num_simulations: int = 50):
        shallow_cfg = copy.copy(CFG)
        shallow_cfg.num_simulations = num_simulations
        self.mcts = MCTS(net, shallow_cfg)

    def select_move(self, board: chess.Board) -> chess.Move:
        root = self.mcts.search(board, add_noise=False)
        return self.mcts.select_move(root, temperature=0.0)


def play_evaluation_game(agent_white, agent_black,
                         max_moves: int = 150,
                         verbose: bool = False) -> str:
    board = chess.Board()
    for move_num in range(max_moves):
        if board.is_game_over():
            break
        agent = agent_white if board.turn == chess.WHITE else agent_black
        move = agent.select_move(board)
        board.push(move)
        if verbose:
            print(f'  {"W" if move_num%2==0 else "B"} move {move_num+1}: {move.uci()}')
    return board.result()


def run_evaluation(net: ChessNet, num_games: int = 10):
    print('\n' + '='*55)
    print('  EVALUATION')
    print('='*55)

    random_agent  = RandomAgent()
    mcts_agent    = MCTSAgent(net, num_simulations=50)

    # ─── vs Random ──────────────────────────────────────────────────────────
    print(f'\n  🎯 MCTS Agent vs Random Agent ({num_games} games)...')
    wins = draws = losses = 0
    for i in range(num_games):
        result = play_evaluation_game(mcts_agent, random_agent)
        if   result == '1-0': wins   += 1
        elif result == '0-1': losses += 1
        else:                 draws  += 1
    print(f'  Results (MCTS as White): W={wins} D={draws} L={losses}')
    win_rate = wins / num_games
    print(f'  Win rate vs Random: {win_rate*100:.1f}%')

    # ─── Random vs MCTS ─────────────────────────────────────────────────────
    wins2 = draws2 = losses2 = 0
    for i in range(num_games):
        result = play_evaluation_game(random_agent, mcts_agent)
        if   result == '1-0': losses2 += 1
        elif result == '0-1': wins2   += 1
        else:                 draws2  += 1
    print(f'  Results (MCTS as Black): W={wins2} D={draws2} L={losses2}')

    total_wins = wins + wins2
    print(f'  Overall win rate: {total_wins}/{2*num_games} = {total_wins/(2*num_games)*100:.1f}%')


run_evaluation(net, num_games=5)

## 16. Debugging Tools

In [ ]:
class DebugAnalyser:
    """Post-hoc debugging helpers for inspecting network and MCTS behaviour."""

    def __init__(self, net: ChessNet, cfg: Config):
        self.net  = net
        self.cfg  = cfg
        self.mcts = MCTS(net, cfg)

    def inspect_position(self, fen: str, num_simulations: int = 200):
        """Deep analysis of a single FEN position."""
        board = chess.Board(fen)
        print(f'\n🔍 Inspecting: {fen}')
        print(f'   Turn: {"White" if board.turn else "Black"}')
        print(f'   Legal moves: {board.legal_moves.count()}')
        print(board)

        # Network raw prediction
        probs, value = self.net.predict(board)
        print(f'\n   NN value estimate: {value:+.4f}')

        # Top-5 policy moves
        legal_preds = [(m, probs[move_to_index(m)]) for m in board.legal_moves]
        legal_preds.sort(key=lambda x: x[1], reverse=True)
        print(f'   NN top-5 moves:')
        for move, prob in legal_preds[:5]:
            print(f'     {move.uci():8s}  {prob:.4f}')

        # MCTS analysis
        tmp_cfg = copy.copy(self.cfg)
        tmp_cfg.num_simulations = num_simulations
        mcts = MCTS(self.net, tmp_cfg)
        root = mcts.search(board, add_noise=False)

        print(f'\n   MCTS top-5 (by visit count) after {num_simulations} sims:')
        children_sorted = sorted(root.children.items(),
                                 key=lambda x: x[1].N, reverse=True)
        for move, child in children_sorted[:5]:
            print(f'     {move.uci():8s}  N={child.N:5d}  Q={child.Q:+.4f}  P={child.P:.4f}')

        # Entropy
        policy_target = mcts.get_policy_target(root, temperature=1.0)
        non_zero = policy_target[policy_target > 0]
        entropy = -np.sum(non_zero * np.log(non_zero + 1e-10))
        print(f'\n   MCTS policy entropy: {entropy:.4f}')
        if entropy < self.cfg.entropy_warn_threshold:
            print(f'   ⚠️  Low entropy — policy may be collapsing')

    def check_gradient_flow(self):
        """Verify gradients are flowing through all layers."""
        print('\n🔬 Gradient flow check:')
        board = chess.Board()
        state = encode_board(board)
        x = torch.tensor(state).unsqueeze(0).to(DEVICE)
        target_p = torch.zeros(1, CFG.policy_size).to(DEVICE)
        target_p[0, 0] = 1.0
        target_v = torch.tensor([[0.5]]).to(DEVICE)

        self.net.train()
        self.net.zero_grad()
        p, v = self.net(x)
        loss = -(target_p * F.log_softmax(p, dim=1)).sum() + F.mse_loss(v, target_v)
        loss.backward()

        for name, param in self.net.named_parameters():
            if param.grad is not None:
                gn = param.grad.norm().item()
                status = '✅' if gn > 1e-8 else '⚠️  DEAD'
                print(f'  {status} {name:45s}  grad_norm={gn:.6f}')
            else:
                print(f'  ❌ NO GRAD  {name}')

    def analyse_replay_buffer(self, replay: ReplayBuffer):
        """Inspect value distribution and policy diversity in the replay buffer."""
        if len(replay) == 0:
            print('Replay buffer is empty.')
            return
        states, policies, values = replay.sample(min(len(replay), 1000))
        print(f'\n📊 Replay buffer analysis ({len(replay)} total samples):')
        print(f'   Value distribution:')
        print(f'     mean={values.mean():.4f}  std={values.std():.4f}  '
              f'min={values.min():.4f}  max={values.max():.4f}')
        unique_wins  = (values > 0.5).sum()
        unique_draws = (np.abs(values) < 0.1).sum()
        unique_loss  = (values < -0.5).sum()
        print(f'     Win positions: {unique_wins}  '
              f'Draw positions: {unique_draws}  '
              f'Loss positions: {unique_loss}')

        policy_entropies = []
        for pol in policies[:100]:
            nz = pol[pol > 0]
            policy_entropies.append(-np.sum(nz * np.log(nz + 1e-10)))
        print(f'   Policy entropy: mean={np.mean(policy_entropies):.4f}  '
              f'min={np.min(policy_entropies):.4f}  '
              f'max={np.max(policy_entropies):.4f}')


# ─── Run debugging ───────────────────────────────────────────────────────────
debugger = DebugAnalyser(net, CFG)

# Gradient flow
debugger.check_gradient_flow()

# Replay buffer analysis
debugger.analyse_replay_buffer(replay)

# Inspect opening position
debugger.inspect_position(chess.STARTING_FEN, num_simulations=100)

## 17. Interactive Analysis: Custom Position

In [ ]:
# ── Analyse a mate-in-1 puzzle ───────────────────────────────────────────────
# Change the FEN below to any position you want to inspect
PUZZLE_FEN = '6k1/5ppp/8/8/8/8/5PPP/3R2K1 w - - 0 1'  # simple endgame

debugger.inspect_position(PUZZLE_FEN, num_simulations=200)

# Policy heatmap for that position
puzzle_board = chess.Board(PUZZLE_FEN)
plot_policy_heatmap(net, puzzle_board,
                    save_path=BASE_DIR / 'policy_heatmap_puzzle.png')

## 18. MCTS Visit Distribution Visualisation

In [ ]:
def visualise_mcts_visits(net: ChessNet, fen: str = chess.STARTING_FEN,
                          num_simulations: int = 200,
                          save_path: Optional[Path] = None):
    """Bar chart of MCTS visit counts across legal moves."""
    board = chess.Board(fen)
    cfg_tmp = copy.copy(CFG)
    cfg_tmp.num_simulations = num_simulations
    mcts = MCTS(net, cfg_tmp)
    root = mcts.search(board, add_noise=False)

    children = [(m.uci(), c.N, c.Q)
                for m, c in root.children.items()]
    children.sort(key=lambda x: x[1], reverse=True)
    children = children[:15]  # top 15

    labels = [c[0] for c in children]
    visits = [c[1] for c in children]
    qvals  = [c[2] for c in children]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].bar(labels, visits, color='steelblue', alpha=0.8)
    axes[0].set_title('MCTS Visit Counts (Top 15 moves)', fontweight='bold')
    axes[0].set_ylabel('Visit count N')
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].grid(axis='y', alpha=0.3)

    colors = ['green' if q >= 0 else 'red' for q in qvals]
    axes[1].bar(labels, qvals, color=colors, alpha=0.8)
    axes[1].set_title('MCTS Q-values (Top 15 moves)', fontweight='bold')
    axes[1].set_ylabel('Q(s, a)')
    axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].grid(axis='y', alpha=0.3)

    plt.suptitle(f'MCTS Analysis — {num_simulations} simulations', fontsize=12)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=100, bbox_inches='tight')
        print(f'  📊 MCTS visit chart saved: {save_path}')
    plt.show()
    plt.close()


visualise_mcts_visits(net, chess.STARTING_FEN, num_simulations=150,
                      save_path=BASE_DIR / 'mcts_visits_opening.png')

## 19. Final Summary

In [ ]:
print('\n' + '█'*60)
print('  TRAINING SUMMARY')
print('█'*60)
print(f'  Total iterations completed : {CFG.num_iterations}')
print(f'  Total self-play games      : {len(all_games)}')
print(f'  Replay buffer size         : {len(replay)}')
total_positions = sum(g.game_length for g in all_games)
print(f'  Total positions generated  : {total_positions:,}')

if logger.history.get('total_loss'):
    losses = logger.history['total_loss']
    print(f'  Initial total loss         : {losses[0]:.4f}')
    print(f'  Final   total loss         : {losses[-1]:.4f}')

print(f'\n  Output directory: {BASE_DIR}')
print(f'  Files saved:')
for f in sorted(BASE_DIR.rglob('*')):
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        print(f'    {f.relative_to(BASE_DIR)}  ({size_kb:.1f} KB)')

print('\n✅ All done!')